# Gemini vs TinyLlama Mortgage Q&A Comparison

## Objective
Compares Gemini and TinyLlama on mortgage question-answering using OCR/PDF extraction, BM25 retrieval, and keyword-based scoring.

## Approach
- Extract mortgage PDF text with OCR fallback
- Chunk and retrieve context with BM25
- Answer the same queries with Gemini and TinyLlama
- Score answers against simple ground-truth terms

## Expected Result
This notebook evaluates model quality and latency tradeoffs for mortgage document question answering.

## Running in Google Colab
These notebooks were developed in Google Colab. For reproducibility, place required PDFs in a `data/` folder when running locally, or upload them to the Colab working directory. The helper functions below try common Colab and GitHub-style paths.

## Security Note
API keys are not stored in the notebook. Use Colab Secrets with the name `GOOGLE_API_KEY` or set the environment variable manually.

## Project Context
This notebook is part of a curated document intelligence externship portfolio project completed through Outamation. The work focuses on OCR, document parsing, retrieval, LLM-based question answering, and prototype application development for mortgage-style document analysis.

## Data Note
The notebooks were originally developed in Google Colab. Any document files used for testing should be placed in the `data/` folder or uploaded directly into the Colab runtime. The sample documents used for this educational project do not contain sensitive personal information.


In [ ]:
# =========================
# Gemini vs TinyLlama on Mortgage Queries
# =========================

# Install dependencies
!pip -q install pymupdf rank-bm25 google-genai transformers accelerate sentencepiece pytesseract pillow
!apt-get -qq install -y tesseract-ocr > /dev/null

import os
import re
import time
import fitz
import torch
import pandas as pd
import pytesseract

from PIL import Image
from rank_bm25 import BM25Okapi
from google import genai
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# PORTABLE COLAB/GITHUB HELPERS
# -------------------------
from pathlib import Path
import os

def resolve_path(filename_or_path):
    """Find a file in common Colab and GitHub project locations."""
    candidates = [
        Path(filename_or_path),
        Path("/content") / filename_or_path,
        Path("data") / Path(filename_or_path).name,
        Path("/content/data") / Path(filename_or_path).name,
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    # Return GitHub-style path as the default so users know where to place data.
    return str(Path("data") / Path(filename_or_path).name)

def get_google_api_key():
    """Load GOOGLE_API_KEY from Colab Secrets or environment variables."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return os.getenv("GOOGLE_API_KEY")

# -------------------------
# SETTINGS
# -------------------------
GEMINI_API_KEY = get_google_api_key()
PDF_PATH = resolve_path("MTG_10009588.pdf")
RUN_TINYLLAMA = True

GEMINI_MODEL_NAME = "gemini-2.5-flash"
LOCAL_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# -------------------------
# QUERIES
# -------------------------
queries = [
    "Who is the borrower?",
    "What is the loan amount?",
    "What is the property address?",
    "What payments is the borrower required to make?",
    "What is included in the monthly payment?",
    "What obligations does the borrower have regarding the property?",
    "What happens if the borrower fails to make payments?",
    "Does this document mention late fees or penalties?",
    "Summarize the key terms of this mortgage."
]

# -------------------------
# SIMPLE GROUND TRUTH SCORING
# -------------------------
ground_truth = {
    "Who is the borrower?": ["kimberly hogan"],
    "What is the loan amount?": ["112,084.00", "$112,084.00", "112084.00"],
    "What is the property address?": ["6468 south 20th street", "milwaukee", "wisconsin", "53221"],
    "What payments is the borrower required to make?": ["principal", "interest", "taxes", "insurance"],
    "What is included in the monthly payment?": ["principal", "interest", "taxes", "insurance", "escrow"],
    "What obligations does the borrower have regarding the property?": ["property", "insurance", "charges", "lender"],
    "What happens if the borrower fails to make payments?": ["default", "lender", "security instrument", "note"],
    "Does this document mention late fees or penalties?": ["late", "charge", "note"],
    "Summarize the key terms of this mortgage.": ["kimberly hogan", "112,084.00", "6468 south 20th street", "mers"]
}

# -------------------------
# PDF EXTRACTION WITH OCR FALLBACK
# -------------------------
def extract_text_from_pdf(pdf_path: str) -> str:
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    doc = fitz.open(pdf_path)
    pages_text = []

    for i, page in enumerate(doc):
        text = page.get_text("text").strip()

        # OCR fallback for scanned pages
        if not text:
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img)

        if text.strip():
            pages_text.append(f"--- PAGE {i+1} ---\n{text}")

    doc.close()
    return "\n\n".join(pages_text).strip()

def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

# -------------------------
# SIMPLE CHUNKING + BM25
# -------------------------
def chunk_text(text: str, chunk_size: int = 800):
    text = text.strip()
    if not text:
        return []
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

def tokenize(text: str):
    return re.findall(r"\w+|\$[\d,\.]+", text.lower())

def build_bm25(chunks):
    tokenized_chunks = [tokenize(chunk) for chunk in chunks]
    tokenized_chunks = [tokens for tokens in tokenized_chunks if tokens]
    if not tokenized_chunks:
        return None
    return BM25Okapi(tokenized_chunks)

def retrieve_chunks(query: str, bm25, chunks, top_k: int = 2):
    if bm25 is None or not chunks:
        return []

    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [chunks[i] for i in ranked_indices]

# -------------------------
# PROMPT BUILDER
# -------------------------
def build_prompt(query: str, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    return f"""You are answering questions about a mortgage document.

Use only the context below.
If the answer is not clearly stated, say that it is not clearly stated.

Context:
{context}

Question: {query}

Answer briefly and clearly:"""

# -------------------------
# GEMINI
# -------------------------
GEMINI_API_KEY = get_google_api_key()

gemini_client = None
if GEMINI_API_KEY:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)

def ask_gemini(prompt: str):
    if gemini_client is None:
        return "[Gemini skipped: add API key]", 0.0

    start = time.time()
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt
    )
    elapsed = round(time.time() - start, 2)
    text = response.text.strip() if getattr(response, "text", None) else "[No response]"
    return text, elapsed

# -------------------------
# TINYLLAMA
# -------------------------
tiny_tokenizer = None
tiny_model = None

if RUN_TINYLLAMA:
    print("Loading TinyLlama...")
    tiny_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)

    if tiny_tokenizer.pad_token is None:
        tiny_tokenizer.pad_token = tiny_tokenizer.eos_token

    tiny_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )

    if getattr(tiny_model.config, "pad_token_id", None) is None:
        tiny_model.config.pad_token_id = tiny_tokenizer.pad_token_id

def ask_tinyllama(prompt: str):
    if tiny_model is None or tiny_tokenizer is None:
        return "[TinyLlama skipped]", 0.0

    start = time.time()

    inputs = tiny_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(tiny_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = tiny_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tiny_tokenizer.pad_token_id
        )

    decoded = tiny_tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()
    elapsed = round(time.time() - start, 2)
    return answer, elapsed

# -------------------------
# SCORING
# -------------------------
def score_answer(query: str, answer: str) -> int:
    expected_terms = ground_truth.get(query, [])
    answer_lower = answer.lower()
    return sum(1 for term in expected_terms if term.lower() in answer_lower)

def verdict(query: str, hits: int) -> str:
    total = max(1, len(ground_truth.get(query, [])))
    ratio = hits / total
    if ratio >= 0.75:
        return "Correct"
    if ratio >= 0.35:
        return "Partial"
    return "Weak"

# -------------------------
# RUN
# -------------------------
print("Extracting text from PDF...")
raw_text = extract_text_from_pdf(PDF_PATH)
text = clean_text(raw_text)

print("\nExtracted text length:", len(text))
print("\nPreview:\n")
print(text[:1500] if text else "No text extracted.")

chunks = chunk_text(text, chunk_size=800)
bm25 = build_bm25(chunks)

if bm25 is None:
    raise ValueError("No usable text was extracted from the PDF.")

print("\nTotal chunks:", len(chunks))

results = []

for query in queries:
    retrieved = retrieve_chunks(query, bm25, chunks, top_k=2)
    prompt = build_prompt(query, retrieved)

    gemini_answer, gemini_time = ask_gemini(prompt)
    tiny_answer, tiny_time = ask_tinyllama(prompt)

    gemini_hits = score_answer(query, gemini_answer)
    tiny_hits = score_answer(query, tiny_answer)

    results.append({
        "Query": query,
        "Gemini Time (s)": gemini_time,
        "Gemini Score": gemini_hits,
        "Gemini Verdict": verdict(query, gemini_hits),
        "TinyLlama Time (s)": tiny_time,
        "TinyLlama Score": tiny_hits,
        "TinyLlama Verdict": verdict(query, tiny_hits),
        "Gemini Answer": gemini_answer,
        "TinyLlama Answer": tiny_answer,
        "Retrieved Context": " || ".join(retrieved)
    })

df = pd.DataFrame(results)

print("\n===== SUMMARY TABLE =====\n")
display(df[[
    "Query",
    "Gemini Time (s)", "Gemini Score", "Gemini Verdict",
    "TinyLlama Time (s)", "TinyLlama Score", "TinyLlama Verdict"
]])

print("\n===== SAMPLE ANSWERS =====\n")
for _, row in df.iterrows():
    print("=" * 100)
    print("QUERY:", row["Query"])
    print("\nGemini:", row["Gemini Answer"])
    print("\nTinyLlama:", row["TinyLlama Answer"])
    print()

csv_path = "outputs/mortgage_comparison_results.csv"
os.makedirs("outputs", exist_ok=True)
df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")
